# SmartBag — Random Forest
Mesmo CSV e split da MLP. Exporta o `.pkl` para a API e C++ com micromlgen. Não há dados ou modelo pré-treinados neste notebook.


## 1. Pacotes


In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn joblib micromlgen==1.1.28


In [ ]:
import json, hashlib, zipfile
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from micromlgen import port


## 2. Abrir e conferir o CSV


In [ ]:
from google.colab import files
arquivos = files.upload()
ARQUIVO = next(iter(arquivos))
df = pd.read_csv(ARQUIVO).sort_values(["rodada", "situacao", "timestamp"]).reset_index(drop=True)
FEATURES = ["temperatura", "umidade", "delta_distancia", "luz", "mov_max", "incl_max"]
CLASSES = ["ENTREGA_OK", "REVISAR_ENTREGA"]
X = df[FEATURES].astype(np.float32)
y = df["target"].map({nome: i for i, nome in enumerate(CLASSES)})
assert y.notna().all(), "Confira os targets do CSV."
assert df["device"].nunique() == 1, "Use uma execução de uma equipe."
display(pd.crosstab(df["rodada"], df["target"]))


## 3. Separar por rodada
A última rodada fica no teste, como no app17-7. RF e MLP usam o mesmo CSV e a mesma divisão.


In [ ]:
rodada_teste = df["rodada"].max()
indices_treino = df.index[df["rodada"] != rodada_teste]
indices_teste = df.index[df["rodada"] == rodada_teste]
X_treino, X_teste = X.loc[indices_treino], X.loc[indices_teste]
y_treino, y_teste = y.loc[indices_treino], y.loc[indices_teste]
rodadas_treino = sorted(df.loc[indices_treino, "rodada"].unique().tolist())
rodadas_teste = [int(rodada_teste)]
assert set(y_treino) == set(y_teste) == {0, 1}, "Colete pelo menos duas rodadas completas com as duas classes."
print("Treino:", rodadas_treino, "| Teste:", rodadas_teste)


## 4. Treinar
20 árvores, sem limitar a profundidade. Isso permite folhas puras, necessárias para os votos do micromlgen coincidirem com as probabilidades do scikit-learn. O teste de exportação confere essa condição.


In [ ]:
modelo = RandomForestClassifier(n_estimators=20, random_state=42)
modelo.fit(X_treino, y_treino)
predicoes = modelo.predict(X_teste)


## 5. Avaliar


In [ ]:
print("Acurácia:", accuracy_score(y_teste, predicoes))
print(classification_report(y_teste, predicoes, labels=[0, 1], target_names=CLASSES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_teste, predicoes, labels=[0, 1], display_labels=CLASSES, cmap="Blues")
plt.show()
display(pd.crosstab(df.iloc[indices_teste]["situacao"], pd.Series(predicoes, index=y_teste.index, name="predicao")))


In [ ]:
pd.Series(modelo.feature_importances_, index=FEATURES).sort_values().plot.barh(title="Importância das features")
plt.show()


## 6. Comparar com uma feature
Um corte testa o limiar; uma árvore pequena testa uma feature com vários cortes. Se alguma resolver tudo, revise a coleta. Os resultados anteriores do dataset sintético não são metas desta coleta.


In [ ]:
maioria = DummyClassifier(strategy="most_frequent").fit(X_treino, y_treino)
print("Classe majoritária:", maioria.score(X_teste, y_teste))
resultados = []
for coluna in FEATURES:
    linha = {"feature": coluna}
    for profundidade in [1, 5]:
        simples = DecisionTreeClassifier(max_depth=profundidade, random_state=42)
        simples.fit(X_treino[[coluna]], y_treino)
        linha[f"profundidade_{profundidade}"] = simples.score(X_teste[[coluna]], y_teste)
    resultados.append(linha)
display(pd.DataFrame(resultados))


## 7. Salvar o modelo avaliado


In [ ]:
joblib.dump(modelo, "modelo_smartbag.pkl")
recarregado = joblib.load("modelo_smartbag.pkl")
assert np.array_equal(recarregado.predict(X_teste), predicoes)
PACOTES = ["scikit-learn", "numpy", "pandas", "joblib", "micromlgen"]
metadados = {
    "features": FEATURES, "classes": CLASSES,
    "unidades": ["C", "%", "cm", "RAW (0..4095)", "m/s2", "graus"],
    "rodadas_treino": rodadas_treino, "rodadas_teste": rodadas_teste,
    "csv_sha256": hashlib.sha256(Path(ARQUIVO).read_bytes()).hexdigest(),
    "versoes": {p: version(p) for p in PACOTES},
}
metadados["modelo"] = modelo.get_params()
Path("metadados_rf.json").write_text(json.dumps(metadados, indent=2), encoding="utf-8")


## 8. Exportar C++
A entrada usa valores originais, sem scaler. O teste abaixo compila o header no Colab e compara com o scikit-learn; se falhar, não trate os artefatos como equivalentes.


In [ ]:
for arvore in modelo.estimators_:
    folhas = arvore.tree_.children_left == -1
    contagens = arvore.tree_.value[folhas, 0, :]
    assert (np.count_nonzero(contagens, axis=1) == 1).all(), "Folhas mistas: micromlgen pode mudar a decisão. Inspecione amostras idênticas com targets diferentes; não altere rótulos apenas para exportar."
header = port(modelo)
Path("AIoTRandomForest_micromlgen.hpp").write_text(header, encoding="utf-8")
X_teste.to_csv("entradas_teste.csv", index=False)
np.savetxt("classes_teste.csv", predicoes, fmt="%d", header="classe", comments="")


In [ ]:
import subprocess
codigo = r"""
#include <iostream>
#include <cstdint>
#include "AIoTRandomForest_micromlgen.hpp"
int main() {
    Eloquent::ML::Port::RandomForest modelo;
    float x[6];
    while (std::cin >> x[0] >> x[1] >> x[2] >> x[3] >> x[4] >> x[5])
        std::cout << modelo.predict(x) << "\n";
}
"""
Path("conferir.cpp").write_text(codigo, encoding="utf-8")
subprocess.run(["g++", "-std=c++11", "conferir.cpp", "-o", "conferir"], check=True)
entrada = X_teste.to_csv(index=False, header=False, sep=" ")
saida = subprocess.run(["./conferir"], input=entrada, text=True, capture_output=True, check=True)
classes_cpp = np.fromstring(saida.stdout, sep=" ", dtype=int)
assert np.array_equal(classes_cpp, predicoes), "C++ divergiu: conferir exportador e precisão antes de embarcar."
print("Python e C++ concordam nas", len(classes_cpp), "amostras de teste.")


## 9. Baixar
O ZIP reúne modelo, header, versões e amostras de conferência. Use as versões registradas também na API.


In [ ]:
with zipfile.ZipFile("smartbag_rf.zip", "w") as pacote:
    for nome in ["modelo_smartbag.pkl", "AIoTRandomForest_micromlgen.hpp", "metadados_rf.json", "entradas_teste.csv", "classes_teste.csv"]:
        pacote.write(nome)
files.download("smartbag_rf.zip")
